# Optimized Vanilla BiRNN (Version 2)

This notebook implements an **Optimized** Vanilla Bidirectional RNN (using `nn.RNN`) to improve upon the baseline score of 0.46.

## Key Optimizations
1.  **Architecture**: `nn.RNN` (Strict Vanilla) with **Orthogonal Initialization** to combat vanishing gradients.
2.  **Data Handling**: **PackedSequences** (`pack_padded_sequence`) to prevent the RNN from processing padding zeros.
3.  **Class Imbalance**: **Weighted CrossEntropyLoss** to better learn rare classes.
4.  **Training Stability**: Gradient Clipping & Learning Rate Scheduler.
5.  **Checkpointing**: Evaluating and saving the **Best Model** based on Validation F1.

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# --- Configuration ---
class Config:
    seed = 42
    device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
    batch_size = 64
    embed_dim = 128
    hidden_dim = 256
    num_layers = 2
    learning_rate = 0.001
    epochs = 15 # Increased slightly as we have scheduler
    
    # Paths
    train_path = "sep-25-dl-gen-ai-nppe-2/train.csv"
    test_path = "sep-25-dl-gen-ai-nppe-2/test.csv"
    
    # TrackIO
    trackio_project = "25-t3-nppe2"
    trackio_space_id = "manyakaistha/dlgenai-nppe"
    trackio_group = "vanilla_birnn_opt" 

cfg = Config()
print(f"Using device: {cfg.device}")

# Set seeds
def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(cfg.seed)

# --- TrackIO Setup ---
os.environ["HUGGINGFACEHUB_API_TOKEN"] = "hf_placeholder_token_for_reproducibility"
os.environ["TRACKIO_SPACE_ID"] = cfg.trackio_space_id

def maybe_init_trackio():
    try:
        import trackio
        token = os.environ.get("HUGGINGFACEHUB_API_TOKEN") or os.environ.get("HF_TOKEN")
        if cfg.trackio_space_id and token:
            trackio.init(
                project=cfg.trackio_project,
                space_id=cfg.trackio_space_id,
                group=cfg.trackio_group
            )
            trackio.log({"status/started": 1})
            print("✓ TrackIO initialized")
            return trackio
    except Exception as e:
        print(f"TrackIO initialization failed: {e}")
        pass
    return None

trk = maybe_init_trackio()

## Data Preprocessing

In [ ]:
PAD_TOKEN = 0

# 1. Amino Acid Map
AA_CHARS = ['<PAD>', '<UNK>', '*', 'A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y']
AA_TO_ID = {c: i for i, c in enumerate(AA_CHARS)}
ID_TO_AA = {i: c for c, i in AA_TO_ID.items()}

# 2. Q8 Map
Q8_CHARS = ['<PAD>'] + sorted(['B', 'C', 'E', 'G', 'H', 'I', 'S', 'T'])
Q8_TO_ID = {c: i for i, c in enumerate(Q8_CHARS)}
ID_TO_Q8 = {i: c for c, i in Q8_TO_ID.items()}

# 3. Q3 Map
Q3_CHARS = ['<PAD>'] + sorted(['C', 'E', 'H'])
Q3_TO_ID = {c: i for i, c in enumerate(Q3_CHARS)}
ID_TO_Q3 = {i: c for c, i in Q3_TO_ID.items()}

# --- Dataset ---
class ProteinDataset(Dataset):
    def __init__(self, csv_path, mode='train'):
        self.data = pd.read_csv(csv_path)
        self.mode = mode
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        seq_str = row['seq']
        
        seq_ids = [AA_TO_ID.get(c, AA_TO_ID['<UNK>']) for c in seq_str]
        
        item = {
            'seq': torch.tensor(seq_ids, dtype=torch.long),
            'id': row['id']
        }
        
        if self.mode == 'train':
            q8_str = row['sst8']
            q3_str = row['sst3']
            
            q8_ids = [Q8_TO_ID.get(c, 0) for c in q8_str]
            q3_ids = [Q3_TO_ID.get(c, 0) for c in q3_str]
            
            item['q8'] = torch.tensor(q8_ids, dtype=torch.long)
            item['q3'] = torch.tensor(q3_ids, dtype=torch.long)
            
        return item

def collate_fn(batch):
    # Sort by length for pack_padded_sequence (Efficient Packing)
    # Note: Modern PyTorch pack_padded_sequence with enforce_sorted=False handles unsorted, 
    # but sorting is good practice.
    batch.sort(key=lambda x: len(x['seq']), reverse=True)
    
    seqs = [item['seq'] for item in batch]
    ids = [item['id'] for item in batch]
    lengths = torch.tensor([len(s) for s in seqs], dtype=torch.long)
    
    seqs_padded = pad_sequence(seqs, batch_first=True, padding_value=PAD_TOKEN)
    
    batch_out = {
        'seq': seqs_padded,
        'ids': ids,
        'lengths': lengths
    }
    
    if 'q8' in batch[0]:
        q8s = [item['q8'] for item in batch]
        q3s = [item['q3'] for item in batch]
        
        batch_out['q8'] = pad_sequence(q8s, batch_first=True, padding_value=PAD_TOKEN)
        batch_out['q3'] = pad_sequence(q3s, batch_first=True, padding_value=PAD_TOKEN)
        
    return batch_out

train_ds = ProteinDataset(cfg.train_path, mode='train')
# Create Validation Split (Optional but recommended for Checkpointing)
# Since user didn't explicitly ask for split, we will use entire train for training 
# OR split small chunk. Best practice: Use a small val split.
from torch.utils.data import random_split
val_size = int(0.1 * len(train_ds))
train_size = len(train_ds) - val_size
train_subset, val_subset = random_split(train_ds, [train_size, val_size], generator=torch.Generator().manual_seed(cfg.seed))

train_loader = DataLoader(train_subset, batch_size=cfg.batch_size, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_subset, batch_size=cfg.batch_size, shuffle=False, collate_fn=collate_fn)

print(f"Train Size: {len(train_subset)}, Val Size: {len(val_subset)}")

### Class Weighting Calculation

In [ ]:
# Flatten all labels in training set to compute weights
all_q8 = []
all_q3 = []

print("Computing Class Weights...")
# We iterate original df to be faster than subset
df_train = pd.read_csv(cfg.train_path)
for s in df_train['sst8']: all_q8.extend([Q8_TO_ID[c] for c in s])
for s in df_train['sst3']: all_q3.extend([Q3_TO_ID[c] for c in s])

classes_q8 = np.arange(len(Q8_CHARS))
classes_q3 = np.arange(len(Q3_CHARS))

# Calculate weights (skipping padding roughly, compute_class_weight handles existing classes)
# Note: PAD is 0. We usually don't want weight for PAD (it's ignored anyway) but sklearn needs all classes.
# We will manually set PAD weight to 0 later just in case.
weights_q8 = compute_class_weight('balanced', classes=classes_q8, y=all_q8)
weights_q3 = compute_class_weight('balanced', classes=classes_q3, y=all_q3)

weights_q8[PAD_TOKEN] = 0.0
weights_q3[PAD_TOKEN] = 0.0

weight_tensor_q8 = torch.tensor(weights_q8, dtype=torch.float32).to(cfg.device)
weight_tensor_q3 = torch.tensor(weights_q3, dtype=torch.float32).to(cfg.device)

print("Weights Q8:", weights_q8)
print("Weights Q3:", weights_q3)

## Optimized Model Architecture

In [ ]:
class BiRNN_Opt(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers, num_classes_q8, num_classes_q3):
        super(BiRNN_Opt, self).__init__()
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD_TOKEN)
        
        # Optimized Vanilla RNN
        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            nonlinearity='tanh' # Explicit tanh
        )
        
        # Linear Heads
        self.q8_head = nn.Linear(hidden_dim * 2, num_classes_q8)
        self.q3_head = nn.Linear(hidden_dim * 2, num_classes_q3)
        
        self._init_weights()
        
    def _init_weights(self):
        # Orthogonal Initialization for RNN weights
        for name, param in self.rnn.named_parameters():
            if 'weight' in name:
                nn.init.orthogonal_(param)
            elif 'bias' in name:
                nn.init.constant_(param, 0.0)
        
        # Xavier for heads
        nn.init.xavier_uniform_(self.q8_head.weight)
        nn.init.xavier_uniform_(self.q3_head.weight)

    def forward(self, x, lengths):
        # x: [batch, seq_len]
        embed = self.embedding(x)
        
        # PACKING: Optimize RNN pass
        # lengths must be on CPU for pack_padded_sequence
        packed_input = pack_padded_sequence(embed, lengths.cpu(), batch_first=True, enforce_sorted=False)
        
        # RNN Output
        packed_output, _ = self.rnn(packed_input)
        
        # UNPACKING
        output, _ = pad_packed_sequence(packed_output, batch_first=True, total_length=x.size(1))
        
        # Heads
        q8_logits = self.q8_head(output)
        q3_logits = self.q3_head(output)
        
        return q8_logits, q3_logits

model = BiRNN_Opt(
    vocab_size=len(AA_TO_ID),
    embedding_dim=cfg.embed_dim,
    hidden_dim=cfg.hidden_dim,
    num_layers=cfg.num_layers,
    num_classes_q8=len(Q8_TO_ID),
    num_classes_q3=len(Q3_TO_ID)
).to(cfg.device)

print(model)

## Training Routine with Checkpointing

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=cfg.learning_rate)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

# Weighted Loss
criterion_q8 = nn.CrossEntropyLoss(weight=weight_tensor_q8, ignore_index=PAD_TOKEN)
criterion_q3 = nn.CrossEntropyLoss(weight=weight_tensor_q3, ignore_index=PAD_TOKEN)

def calculate_metric(preds, targets, pad_idx=0):
    preds_flat = preds.argmax(dim=-1).view(-1).cpu().numpy()
    targets_flat = targets.view(-1).cpu().numpy()
    mask = targets_flat != pad_idx
    return f1_score(targets_flat[mask], preds_flat[mask], average='macro')

def train_epoch(model, loader, optimizer, device, epoch):
    model.train()
    total_loss = 0
    total_q8_f1 = 0
    total_q3_f1 = 0
    count = 0
    
    pbar = tqdm(loader, desc=f"Epoch {epoch+1} Train")
    for batch in pbar:
        seq = batch['seq'].to(device)
        lengths = batch['lengths']
        q8_tgt = batch['q8'].to(device)
        q3_tgt = batch['q3'].to(device)
        
        optimizer.zero_grad()
        
        q8_logits, q3_logits = model(seq, lengths)
        
        loss_q8 = criterion_q8(q8_logits.view(-1, len(Q8_TO_ID)), q8_tgt.view(-1))
        loss_q3 = criterion_q3(q3_logits.view(-1, len(Q3_TO_ID)), q3_tgt.view(-1))
        loss = loss_q8 + loss_q3
        
        loss.backward()
        
        # Gradient Clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        
        optimizer.step()
        
        total_loss += loss.item()
        f1_q8 = calculate_metric(q8_logits.detach(), q8_tgt, PAD_TOKEN)
        f1_q3 = calculate_metric(q3_logits.detach(), q3_tgt, PAD_TOKEN)
        total_q8_f1 += f1_q8
        total_q3_f1 += f1_q3
        count += 1
        
        pbar.set_postfix({'loss': loss.item(), 'f1_q8': f1_q8})
        
        if trk:
            trk.log({'train/loss': loss.item(), 'train/f1_q8': f1_q8, 'train/f1_q3': f1_q3})
            
    return total_loss/count, total_q8_f1/count, total_q3_f1/count

def evaluate(model, loader, device):
    model.eval()
    total_loss = 0
    total_q8_f1 = 0
    total_q3_f1 = 0
    count = 0
    
    with torch.no_grad():
        for batch in loader:
            seq = batch['seq'].to(device)
            lengths = batch['lengths']
            q8_tgt = batch['q8'].to(device)
            q3_tgt = batch['q3'].to(device)
            
            q8_logits, q3_logits = model(seq, lengths)
            
            loss_q8 = criterion_q8(q8_logits.view(-1, len(Q8_TO_ID)), q8_tgt.view(-1))
            loss_q3 = criterion_q3(q3_logits.view(-1, len(Q3_TO_ID)), q3_tgt.view(-1))
            loss = loss_q8 + loss_q3
            
            total_loss += loss.item()
            total_q8_f1 += calculate_metric(q8_logits, q8_tgt, PAD_TOKEN)
            total_q3_f1 += calculate_metric(q3_logits, q3_tgt, PAD_TOKEN)
            count += 1
            
    return total_loss/count, total_q8_f1/count, total_q3_f1/count

# --- Main Loop ---
best_score = 0.0

for epoch in range(cfg.epochs):
    train_loss, train_f1_8, train_f1_3 = train_epoch(model, train_loader, optimizer, cfg.device, epoch)
    val_loss, val_f1_8, val_f1_3 = evaluate(model, val_loader, cfg.device)
    
    # Harmonic Mean Score
    val_score = (2 * val_f1_8 * val_f1_3) / (val_f1_8 + val_f1_3 + 1e-8)
    
    scheduler.step(val_loss)
    
    print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f} | Val Loss={val_loss:.4f} | Val Score={val_score:.4f} (F1_Q8={val_f1_8:.4f}, F1_Q3={val_f1_3:.4f})")
    
    if trk:
        trk.log({'val/loss': val_loss, 'val/score': val_score})
    
    # Checkpoint
    if val_score > best_score:
        best_score = val_score
        torch.save(model.state_dict(), 'best_model.pth')
        print("--> Best Model Saved")

print(f"Training Complete. Best Val Score: {best_score:.4f}")

## Inference (Using Best Model)

In [ ]:
test_ds = ProteinDataset(cfg.test_path, mode='test')
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, collate_fn=collate_fn)

# Load Best Model
model.load_state_dict(torch.load('best_model.pth'))
model.eval()
print("Loaded Best Model for Inference.")

results = []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Inference"):
        seq = batch['seq'].to(cfg.device)
        lengths = batch['lengths']
        ids = batch['ids']
        
        q8_logits, q3_logits = model(seq, lengths)
        
        q8_preds = q8_logits.argmax(dim=-1).cpu().numpy()
        q3_preds = q3_logits.argmax(dim=-1).cpu().numpy()
        
        for i, id_ in enumerate(ids):
            length = lengths[i].item()
            pred_q8_ids = q8_preds[i, :length]
            pred_q3_ids = q3_preds[i, :length]
            
            pred_q8_str = ''.join([ID_TO_Q8[idx] for idx in pred_q8_ids])
            pred_q3_str = ''.join([ID_TO_Q3[idx] for idx in pred_q3_ids])
            
            results.append({'id': id_, 'sst8': pred_q8_str, 'sst3': pred_q3_str})

df_sub = pd.DataFrame(results)
df_sub.to_csv('submission_v2.csv', index=False)
print("Saved submission_v2.csv")
df_sub.head()